# Notebook 11: Profiling Distributed Training

## Why This Matters

You cannot optimize what you cannot measure. Distributed training failures are subtle: a 10% slowdown on 1000 GPUs costs $50K/day. The difference between a 40% MFU (model FLOP utilization) and a 60% MFU -- achievable with proper profiling -- can cut training costs by a third. This notebook covers the PyTorch Profiler, NVIDIA Nsight, and the key metrics every ML engineer needs to know: MFU, GPU utilization, communication overlap, and bubble fraction.

In [ ]:
%pip install -q torch numpy matplotlib

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import time
import json

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")
print("Ready.")

## 1. Model FLOP Utilization (MFU): The Key Metric

**MFU** (Model FLOP Utilization) measures how efficiently you are using your hardware:

$$\text{MFU} = \frac{\text{Actual FLOP/s achieved}}{\text{Peak theoretical FLOP/s}}$$

For an A100 SXM4: peak FP16/BF16 = 312 TFLOP/s (with sparsity) or 77.6 TFLOP/s (dense).

Computing achieved FLOP/s from training:
1. Count FLOPs per forward+backward pass: `6 × N × B × seq` (for transformers)
2. Divide by step time

A "good" MFU for LLM training:
- < 30%: something is wrong (idle GPUs, poor overlap, tiny batches)
- 30-40%: typical for first implementation
- 40-50%: with tuning (proper batch size, bucket sizes, TP/PP config)
- 50%+: excellent, requires FlashAttention, fused kernels, well-tuned pipeline

In [ ]:
# MFU calculation for a transformer model

def count_transformer_flops(n_layers, d_model, n_heads, d_ff, seq_len, batch_size):
    """
    FLOPs for one forward pass through a GPT-style transformer.
    Per token approximate: 2 * N_params (for large d_model).
    
    Breaking it down per layer:
    - Self-attention QKV: 2 * B * seq * d_model * 3*d_model (column-parallel)
    - Attention scores: 2 * B * n_heads * seq * seq * d_head
    - Attention output: 2 * B * seq * d_model * d_model
    - FFN fc1: 2 * B * seq * d_model * d_ff
    - FFN fc2: 2 * B * seq * d_ff * d_model
    """
    d_head = d_model // n_heads
    
    flops_per_layer = (
        # QKV projections
        2 * batch_size * seq_len * d_model * (3 * d_model) +
        # Attention scores (seq * seq per head)
        2 * batch_size * n_heads * seq_len * seq_len * d_head +
        # Output projection
        2 * batch_size * seq_len * d_model * d_model +
        # FFN layer 1
        2 * batch_size * seq_len * d_model * d_ff +
        # FFN layer 2
        2 * batch_size * seq_len * d_ff * d_model
    )
    
    return n_layers * flops_per_layer

def compute_mfu(n_layers, d_model, n_heads, d_ff, seq_len, batch_size, 
                step_time_s, peak_flops_tflops, fwd_bwd_ratio=3):
    """
    fwd_bwd_ratio=3: backward is ~2x forward, total = 3x forward FLOPs
    """
    fwd_flops = count_transformer_flops(n_layers, d_model, n_heads, d_ff, seq_len, batch_size)
    total_flops = fwd_flops * fwd_bwd_ratio
    achieved_tflops = total_flops / step_time_s / 1e12
    mfu = achieved_tflops / peak_flops_tflops
    return achieved_tflops, mfu

# Example: 7B parameter model
n_layers, d_model, n_heads, d_ff = 32, 4096, 32, 11008  # LLaMA-7B approx
seq_len, batch_size = 2048, 4
peak_flops = 77.6  # A100 dense BF16 TFLOP/s

# Simulate different step times (representing different optimizations)
print(f"7B model (LLaMA-7B config), seq={seq_len}, batch={batch_size}")
print(f"Peak: {peak_flops} TFLOP/s (A100 dense BF16)")
print()

fwd_flops = count_transformer_flops(n_layers, d_model, n_heads, d_ff, seq_len, batch_size)
print(f"FLOPs per forward pass: {fwd_flops / 1e12:.1f} TFLOP")
print(f"FLOPs per step (fwd+bwd): {fwd_flops * 3 / 1e12:.1f} TFLOP")
print()

scenarios = [
    ("Unoptimized (MFU~20%)", 0.060),
    ("Standard DDP (MFU~35%)", 0.034),
    ("With FlashAttn (MFU~45%)", 0.027),
    ("Fully optimized (MFU~55%)", 0.022),
]

print(f"{'Scenario':<35} {'Step time':<12} {'TFLOP/s':<12} {'MFU'}")
print("-" * 68)
for name, step_t in scenarios:
    tflops, mfu = compute_mfu(n_layers, d_model, n_heads, d_ff, seq_len, batch_size, 
                               step_t, peak_flops)
    print(f"{name:<35} {step_t*1000:<12.0f}ms {tflops:<12.1f} {mfu:.1%}")

## 2. PyTorch Profiler

The PyTorch Profiler captures CPU/GPU timeline data including:
- CUDA kernel names and durations
- Memory allocations and deallocations
- Synchronization events
- CPU-GPU overlap

Key patterns to look for:
- **Long cudaDeviceSynchronize calls**: CPU is waiting for GPU -- computation-communication not overlapping
- **Small kernel launches**: overhead-dominated, need kernel fusion
- **Large gaps between kernels**: GPU idle, likely waiting for data from CPU
- **Memory fragmentation**: many small allocations causing OOM

In [ ]:
# PyTorch Profiler walkthrough
from torch.profiler import profile, record_function, ProfilerActivity

class SimpleTransformerLayer(nn.Module):
    def __init__(self, d=256, n_heads=8, d_ff=1024):
        super().__init__()
        self.norm1 = nn.LayerNorm(d)
        self.norm2 = nn.LayerNorm(d)
        self.qkv = nn.Linear(d, 3 * d)
        self.out = nn.Linear(d, d)
        self.ff1 = nn.Linear(d, d_ff)
        self.ff2 = nn.Linear(d_ff, d)
    
    def forward(self, x):
        with record_function("attention"):
            B, S, D = x.shape
            qkv = self.qkv(self.norm1(x))
            q, k, v = qkv.chunk(3, dim=-1)
            q = q.view(B, S, 8, D//8).transpose(1, 2)
            k = k.view(B, S, 8, D//8).transpose(1, 2)
            v = v.view(B, S, 8, D//8).transpose(1, 2)
            attn = torch.softmax(q @ k.transpose(-2, -1) / (D//8)**0.5, dim=-1) @ v
            attn = attn.transpose(1, 2).contiguous().view(B, S, D)
            x = x + self.out(attn)
        
        with record_function("ffn"):
            x = x + self.ff2(torch.relu(self.ff1(self.norm2(x))))
        return x

model = SimpleTransformerLayer().to(device)
x = torch.randn(2, 64, 256, device=device)

# Profile with PyTorch profiler
activities = [ProfilerActivity.CPU]
if device == 'cuda':
    activities.append(ProfilerActivity.CUDA)

with profile(
    activities=activities,
    record_shapes=True,
    profile_memory=True,
    with_stack=False,
) as prof:
    for _ in range(3):
        with record_function("full_step"):
            out = model(x)
            loss = out.mean()
            loss.backward()
            model.zero_grad()

# Print key statistics
print("Top operations by CPU time:")
print(prof.key_averages().table(
    sort_by="cpu_time_total",
    row_limit=10,
))

## 3. Key Performance Indicators for Distributed Training

Beyond MFU, track these metrics:

| Metric | How to measure | Warning threshold |
|--------|---------------|-------------------|
| **MFU** | FLOPs / (step_time * peak_FLOPS) | < 30% |
| **GPU utilization** | `nvidia-smi` SM activity | < 80% |
| **Communication/compute ratio** | NCCL time / total step time | > 20% |
| **Bubble fraction** (PP) | Idle time / total step time | > 10% |
| **Data loading stall** | Time waiting for DataLoader | > 5% |
| **Kernel launch overhead** | CPU time not matching GPU time | > 10ms/step |
| **Memory fragmentation** | Peak alloc vs peak reserved | > 20% diff |

**The profiling workflow**:
1. Measure baseline MFU
2. Profile with PyTorch Profiler to identify top ops
3. Check if bottleneck is compute, memory, or communication
4. Apply fix (fused kernel, larger batch, better overlap)
5. Re-measure MFU

In [ ]:
# Simulate the profiling diagnostic workflow

def simulate_training_profile(step_time_ms, compute_ms, comm_ms, data_ms, overhead_ms):
    """Break down a training step into components."""
    total = compute_ms + comm_ms + data_ms + overhead_ms
    # actual step time might have overlap
    return {
        'step_time': step_time_ms,
        'compute': compute_ms,
        'communication': comm_ms,
        'data_loading': data_ms,
        'kernel_overhead': overhead_ms,
        'comm_compute_ratio': comm_ms / compute_ms,
        'efficiency': compute_ms / step_time_ms,
    }

scenarios = {
    'Unoptimized': simulate_training_profile(400, 180, 150, 40, 30),
    'After DDP bucket tuning': simulate_training_profile(300, 180, 80, 40, 0),
    'After async data loading': simulate_training_profile(270, 180, 80, 10, 0),
    'After fused kernels': simulate_training_profile(230, 160, 60, 10, 0),
    'After full optimization': simulate_training_profile(200, 160, 30, 10, 0),
}

print(f"{'Scenario':<30} {'Step(ms)':<10} {'Efficiency':<12} {'Comm/Compute'}")
print("-" * 65)
for name, p in scenarios.items():
    print(f"{name:<30} {p['step_time']:<10.0f} {p['efficiency']:<12.1%} {p['comm_compute_ratio']:.2f}")

# Visualize the breakdown
fig, axes = plt.subplots(1, len(scenarios), figsize=(16, 5))
components = ['compute', 'communication', 'data_loading', 'kernel_overhead']
colors = ['steelblue', 'tomato', 'orange', 'gray']

for ax, (name, p) in zip(axes, scenarios.items()):
    values = [p[c] for c in components]
    bars = ax.bar(range(len(components)), values, color=colors, alpha=0.8)
    ax.set_xticks(range(len(components)))
    ax.set_xticklabels([c.replace('_', '\n') for c in components], fontsize=7)
    ax.set_ylabel('Time (ms)')
    ax.set_title(name.replace(' ', '\n'), fontsize=8)
    ax.set_ylim(0, 200)
    for bar, val in zip(bars, values):
        if val > 5:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                   f'{val:.0f}', ha='center', va='bottom', fontsize=7)

plt.suptitle('Training Step Breakdown by Component', fontsize=11)
plt.tight_layout()
plt.savefig('training_profile.png', dpi=100)
plt.show()

## 4. Common Performance Anti-Patterns

These are the most frequent causes of poor distributed training performance:

**1. find_unused_parameters=True when not needed**
- Adds a graph traversal on every backward pass
- Can add 10-30% overhead
- Fix: only use when your model has conditional computation

**2. Gradient bucket size too small**
- Many small NCCL all-reduce calls instead of few large ones
- Fix: increase `bucket_cap_mb` from 25 to 250 MB for large models

**3. DistributedSampler with shuffle=False on every epoch**
- All ranks see the same data order
- Fix: call `sampler.set_epoch(epoch)` at the start of each epoch

**4. CPU data preprocessing bottleneck**
- GPU is faster than CPU can supply data
- Fix: increase DataLoader `num_workers`, use `pin_memory=True`, prefetch

**5. Synchronization at every step**
- Calling `loss.item()` or `tensor.cpu()` in the training loop
- Blocks GPU until CPU gets the value
- Fix: accumulate then log every N steps

In [ ]:
# Demonstrate the synchronization overhead anti-pattern

def train_with_sync(model, optimizer, X, y, n_steps=50):
    """BAD: calls .item() every step, causing GPU sync."""
    times = []
    for step in range(n_steps):
        t0 = time.perf_counter()
        optimizer.zero_grad()
        out = model(X)
        loss = nn.CrossEntropyLoss()(out, y)
        loss.backward()
        optimizer.step()
        
        # BAD: this forces GPU->CPU sync every step
        loss_val = loss.item()  # <-- synchronization point
        times.append(time.perf_counter() - t0)
    return times, loss_val

def train_without_sync(model, optimizer, X, y, n_steps=50, log_interval=10):
    """GOOD: accumulate loss, only sync every log_interval steps."""
    times = []
    accumulated_loss = torch.zeros(1, device=X.device)
    for step in range(n_steps):
        t0 = time.perf_counter()
        optimizer.zero_grad()
        out = model(X)
        loss = nn.CrossEntropyLoss()(out, y)
        loss.backward()
        optimizer.step()
        
        # GOOD: accumulate without sync
        with torch.no_grad():
            accumulated_loss += loss.detach()
        
        if (step + 1) % log_interval == 0:
            loss_val = accumulated_loss.item() / log_interval  # sync every 10 steps
            accumulated_loss.zero_()
        
        times.append(time.perf_counter() - t0)
    return times, loss_val

torch.manual_seed(42)
model = nn.Sequential(nn.Linear(64, 256), nn.ReLU(), nn.Linear(256, 10)).to(device)
optimizer1 = torch.optim.AdamW(model.parameters(), lr=1e-3)
optimizer2 = torch.optim.AdamW(model.parameters(), lr=1e-3)

X = torch.randn(128, 64, device=device)
y = torch.randint(0, 10, (128,), device=device)

# Warmup
for _ in range(5):
    out = model(X)
    nn.CrossEntropyLoss()(out, y).backward()
    model.zero_grad()

times_sync, _ = train_with_sync(model, optimizer1, X, y)
times_no_sync, _ = train_without_sync(model, optimizer2, X, y)

print(f"With sync every step:    {np.mean(times_sync)*1000:.3f} ms/step")
print(f"Without sync (every 10): {np.mean(times_no_sync)*1000:.3f} ms/step")
overhead = (np.mean(times_sync) - np.mean(times_no_sync)) / np.mean(times_no_sync)
print(f"Sync overhead: {overhead:.1%}")
print()
print("On GPU this overhead is much larger (GPU must finish current work before CPU reads)")
print("Rule: never call .item() or .cpu() in the training hot path.")

## Summary

### Key Concepts

| Tool | What it measures | Command |
|------|-----------------|---------|
| PyTorch Profiler | CPU/GPU kernel timeline | `torch.profiler.profile(...)` |
| `nvidia-smi` | SM utilization, memory, power | `nvidia-smi dmon -s u` |
| NCCL debug | Communication events | `NCCL_DEBUG=INFO` |
| Nsight Systems | Full system timeline | `nsys profile python train.py` |
| `torch.cuda.memory_stats()` | Detailed memory accounting | Per-tensor memory tracking |

**The MFU checklist**:
1. Measure baseline MFU
2. If < 30%: check data loading, GPU utilization in nvidia-smi
3. If 30-40%: check communication overlap, bucket sizes
4. If 40-50%: add FlashAttention, fused kernels
5. If 50%+: you're doing well; focus on other bottlenecks

**Never profile in isolation**: run the profiler for exactly the same number of steps under the same conditions you care about. One-off measurements can be misleading due to caching, JIT compilation, and memory state.

**Next**: `12_capstone_scaling_laws_and_tradeoffs.ipynb` -- putting it all together with a comprehensive analysis of distributed training decisions.